In [3]:
from typing import List, Dict, Any, TypedDict, Optional
from langgraph.graph import StateGraph, START, END
import json
import os
import getpass
import re
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from pprint import pprint
from pydantic import BaseModel, ValidationError

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  

from schemas.query_check_blueprint import QueryCheckBlueprint
from schemas.planner_blueprint import PlannerBlueprint

from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from IPython.display import Image
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env")

True

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY:\n")
    
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
)

search_agent = create_agent(llm,[tavily_search_tool])

# 1. Query Agent

In [8]:
class State(TypedDict):
    graph_state: str
    user_input: str
    
    query_check: Optional[QueryCheckBlueprint]
    query_facts: List[Dict[str, Any]]
    query_approved: bool
    query_attempts: int
    query_feedback: str
    
    plan: Optional[PlannerBlueprint]

In [9]:
def Query_Agent(state: State) -> State:
    user_input = state["user_input"]
    attempts = state.get("query_attempts", 0)

    query_check_prompt = f"""
You are an expert fact-checking agent.
Your task is to analyse the details provided and identify factual claims.
Instructions:
1. Extract every fact from the details.
2. Check each fact for accuracy using reliable public sources.
3. Evaluate whether the fact is relevant to the occasion described.
4. If a fact refers to information that is unlikely to be publicly available, do NOT attempt to fact-check it. Mark it as supported = null.
5. Return ONLY valid JSON. Following the output format strictly.

Details:
{user_input}

Output format:
{{
  "checks_results": [
    {{
      "serial_number": 1,
      "fact_identified": "<extracted factual claim>",
      "supported": <true | false | null>,
      "relevant": <true | false | null>,
      "source_url": "<URL used to verify the claim, or null if not checked>",
      "feedback": "<optional feedback>"
    }}
  ]
}}
"""

    # Step 1: Run search agent to gather facts with web access
    search_result = search_agent.invoke({
        "messages": [
            {"role": "user", "content": query_check_prompt}
        ]
    })

    # Step 2: Extract last assistant message and parse into structured output
    last_message = search_result["messages"][-1].content
    print("FACT CHECK RESULTS")
    print(last_message)
    last_message = search_result["messages"][-1].content
    parsed_json = json.loads(last_message)
    facts = QueryCheckBlueprint.model_validate(parsed_json)

    # Step 3: Evaluate results
    query_approved = True
    feedback_list = []
    for item in facts.checks_results:
        if item.supported is False or item.relevant is False:
            query_approved = False
            feedback_list.append(
                f"Fact: {item.fact_identified}\n"
                f"Supported: {item.supported}\n"
                f"Relevance: {item.relevant}\n"
                f"Feedback: {item.feedback}"
            )

    query_feedback = "\n\n".join(feedback_list)  # cleaner separator

    return {
        "query_check": facts,
        "query_facts": [f.model_dump() for f in facts.checks_results],
        "query_approved": query_approved,
        "query_attempts": attempts + 1,
        "query_feedback": query_feedback,
        "user_input": user_input,
        "graph_state": state["graph_state"],
    }

In [10]:
def Planner_Agent(state: State) -> dict:
    user_input = state.get("user_input")
    query_facts = state.get("query_facts")

    planner_prompt = f"""
You are an expert speech coach.
Your task:
1. Given verified speech details, produce a structured speech plan.
2. Facts that have undergone fact-checking should be placed under "must_include_facts". All other content should be placed under "must_include_points".
3. estimated_wpm must be an integer between 120 and 150.
4. target_word_count must be calculated using: time_limit_minutes * estimated_wpm.
5. Return ONLY valid JSON. Follow the JSON structure strictly. Do not include explanations or text outside the JSON.

Speech Details: {user_input}
Checked facts: {query_facts}

JSON structure:
{{
  "request": {{
    "topic": "",
    "audience": "",
    "occasion": "",
    "time_limit_minutes": 5
  }},
  "targets": {{
    "estimated_wpm": 130,
    "target_word_count": 650
  }},
  "sections": [
    {{
      "section_id": "S1",
      "name": "",
      "purpose": "",
      "must_include_points": [],
      "must_include_facts": []
    }}
  ]
}}
"""

    response = llm.invoke(planner_prompt)
    print("PLANNER RESPONSE")
    content_str = response.content.strip()
    
    # Robust markdown fence removal
    content_str = re.sub(r"^```(?:json)?\s*\n?", "", content_str)
    content_str = re.sub(r"\n?```$", "", content_str).strip()
    print("CONTENT_STR")
    print(content_str)
    try:
        plan = PlannerBlueprint(**json.loads(content_str))
    except (json.JSONDecodeError, ValidationError) as e:
        raise ValueError(
            f"Planner_Agent failed to parse LLM output: {e}\n"
            f"Raw output:\n{content_str}"
        )

    return {"plan": plan}

In [11]:
def collect_user_feedback(topic, audience, occasion, time_limit_in_minutes) -> str:
    revised_content = input("Revised Content to be included (i.e. Point, Examples and Facts):").strip()
    return (
        f"Topic: {topic}\n"
        f"Audience: {audience}\n"
        f"Occasion: {occasion}\n"
        f"Time limit (in minutes): {time_limit_in_minutes}\n\n"
        f"Revised Content:\n{revised_content}"
    )

In [12]:
def Human_Feedback(state: State) -> dict:
    print("\n" + "=" * 50)
    print("FACT-CHECK FAILED — Amendment required")
    print("=" * 50)
    print(f"{state.get('query_feedback', 'No feedback provided.')}")
    print()
    print("Please re-enter your speech content with the issues fixed.")

    # Extract existing values from state
    existing_input = state.get("user_input", "")
    parsed = {}
    for line in existing_input.splitlines():
        for key in ["Topic", "Audience", "Occasion", "Time limit (in minutes)"]:
            if line.startswith(f"{key}:"):
                parsed[key] = line[len(f"{key}:"):].strip()

    updated_input = collect_user_feedback(
        topic=parsed.get("Topic", ""),
        audience=parsed.get("Audience", ""),
        occasion=parsed.get("Occasion", ""),
        time_limit_in_minutes=parsed.get("Time limit (in minutes)", ""),
    )

    return {
        "user_input":     updated_input,
        "graph_state":    updated_input,
        "query_approved": False,
        "query_feedback": "",
    }

In [13]:
def route_user(state):
    approved = state.get("query_approved") or state.get("query_attempts", 0) >= 2
    return "approved" if approved else "rejected"

In [ ]:
from langgraph.graph import StateGraph, START, END
from graph.state import SpeechScriptState, route_user, route_ted, route_content, route_style
from agents.query_agent import Query_Agent
from agents.planner_agent import Planner_Agent
from agents.ted_agent import ted_agent_node
from agents.structure_checking_agent import structure_checking_agent_node
from agents.ted_revision_agent import ted_revision_agent_node
# from agents.content_agent import Content_Agent
# from agents.stylistic_agent import Stylistic_Agent
# from agents.structure_checking_agent import Structure_Checking_Agent
# from agents.grounding_agent import Grounding_Agent
# from agents.reflection_agent import Reflection_Agent
from agents.human_feedback import Human_Feedback, collect_user_feedback
from agents.judging_agent import judging_agent_node



def build_graph():
    builder = StateGraph(SpeechScriptState)

    # Add nodes
    builder.add_node("Query_Agent", Query_Agent)
    builder.add_node("Human_Feedback", Human_Feedback)
    builder.add_node("Planner_Agent", Planner_Agent)
    builder.add_node("TED_Agent", ted_agent_node)
    builder.add_node("Structure_Checking_Agent", structure_checking_agent_node)
    builder.add_node("ted_revision_agent", ted_revision_agent_node)
    # builder.add_node("Content_Agent", Content_Agent)
    # builder.add_node("Grounding_Agent", Grounding_Agent)
    # builder.add_node("Stylistic_Agent", Stylistic_Agent)
    # builder.add_node("Reflection_Agent", Reflection_Agent)
    builder.add_node("judging_agent", judging_agent_node)

    # Define flow
    builder.add_edge(START, "Query_Agent")
    builder.add_edge("Human_Feedback", "Query_Agent") 
    builder.add_conditional_edges(
        "Query_Agent",
        route_user,
        {
            "approved": "Planner_Agent",
            "rejected": "Human_Feedback" # HITL - needs to go back to user with feedback
        }
    )
    builder.add_edge("Planner_Agent", END)
    
    # builder.add_edge("Planner_Agent", "TED_Agent")

    # # Loop 1: TED <> Structure Checker
    # builder.add_edge("TED_Agent", "Structure_Checking_Agent")
    # builder.add_conditional_edges(
    #     "Structure_Checking_Agent",
    #     route_ted,
    #     {
    #         "approved": "Content_Agent",
    #         "rejected": "TED_Agent",
    #     },
    # )

    # # Loop 2: Content <> Grounding
    # builder.add_edge("Content_Agent", "Grounding_Agent")
    # builder.add_conditional_edges(
    #     "Grounding_Agent",
    #     route_content,
    #     {
    #         "approved": "Stylistic_Agent",
    #         "rejected": "Content_Agent",
    #     },
    # )

    # # Loop 3: Stylistic <> Reflection
    # builder.add_edge("Stylistic_Agent", "Reflection_Agent")
    # builder.add_conditional_edges(
    #     "Reflection_Agent",
    #     route_style,
    #     {
    #         "approved": END,
    #         "rejected": "Stylistic_Agent",
    #     },
    # )

    # Compile graph
    return builder.compile()

graph = build_graph()
# Compile graph 
graph = builder.compile()

# Visualize the graph 
Image(graph.get_graph().draw_mermaid_png())
        